# Punto 3 · Anatomía de Stable Diffusion: cómo interactúan sus componentes

**Universidad Autónoma de Occidente** · Inteligencia Artificial Generativa · Ingeniería de Datos e IA
Profesores: Jesús Alfonso López – José Luis Paniagua

**Objetivo:** abrir Stable Diffusion v1.5 pieza por pieza, extraer **a mano** los tensores que se pasan entre sus componentes y visualizar:

1. La diferencia entre el **espacio latente (4×64×64)** donde ocurre la difusión y el **espacio de píxeles (3×512×512)** donde vemos la imagen.
2. El **proceso de generación** paso a paso, tanto en el latente como decodificado a píxeles.
3. Cómo el **texto** guía a la imagen (embeddings de CLIP, mapas de atención cruzada y *classifier-free guidance*).

Usamos la biblioteca **`diffusers`** de Hugging Face. En lugar de llamar al *pipeline* completo con una sola línea, cargamos cada componente por separado y escribimos el ciclo de generación nosotros mismos.

```
                   ┌──────────────────────┐
 "prompt" ─► Tokenizer ─► CLIP Text Encoder ─► embeddings (1×77×768) ──┐
                   └──────────────────────┘                             │ atención cruzada
                                                                        ▼
 ruido z_T (1×4×64×64) ─► [ U-Net predice ε  ─►  Scheduler da un paso ] × N ─► z_0 (1×4×64×64)
                                ▲                                                     │
                                └────────────── t (paso de tiempo) ───────────────────┘
                                                                                      ▼
                                                          VAE Decoder ─► imagen (1×3×512×512)
```

> ⚙️ **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**. El modelo pesa ~2 GB en fp16; la primera descarga tarda unos minutos.

## 0. Instalación e importaciones

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate

In [ ]:
import math, time, io
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import AutoencoderKL, UNet2DConditionModel, DDIMScheduler
from diffusers.models.attention_processor import AttnProcessor
from transformers import CLIPTokenizer, CLIPTextModel

device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if device == "cuda" else torch.float32
torch.set_grad_enabled(False)          # sólo inferencia: no necesitamos gradientes
print("Dispositivo:", device, "| precisión:", DTYPE)
if device == "cpu":
    print("⚠️ Sin GPU esto será muy lento. Activa la GPU en Colab.")

## 1. Cargar los componentes por separado

Stable Diffusion no es una sola red, sino **cuatro piezas** que trabajan juntas:

| Componente | Clase en `diffusers` | Qué hace |
|---|---|---|
| **Tokenizer** | `CLIPTokenizer` | Parte el texto en *tokens* (BPE) y lo rellena a 77 posiciones |
| **Text Encoder** | `CLIPTextModel` (ViT-L/14) | Convierte los tokens en 77 vectores de 768 dimensiones con significado |
| **U-Net** | `UNet2DConditionModel` | En el espacio latente, predice el ruido de $z_t$ dado $t$ y el texto (vía atención cruzada) |
| **VAE** | `AutoencoderKL` | Comprime imágenes 512×512×3 a latentes 64×64×4 (encoder) y los descomprime (decoder) |
| **Scheduler** | `DDIMScheduler` | No tiene pesos: define el ruido en cada $t$ y cómo pasar de $z_t$ a $z_{t-1}$ |

El VAE lo dejamos en float32 para evitar desbordamientos numéricos al decodificar; el resto en float16 para ahorrar memoria de la GPU.

In [ ]:
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"

def load(cls, subfolder, dtype):
    # Intenta bajar los pesos fp16 (más livianos); si no existen, baja los normales
    try:
        return cls.from_pretrained(MODEL_ID, subfolder=subfolder, torch_dtype=dtype, variant="fp16")
    except Exception:
        return cls.from_pretrained(MODEL_ID, subfolder=subfolder, torch_dtype=dtype)

t0 = time.time()
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_encoder = load(CLIPTextModel, "text_encoder", DTYPE).to(device).eval()
vae = load(AutoencoderKL, "vae", torch.float32).to(device).eval()
unet = load(UNet2DConditionModel, "unet", DTYPE).to(device).eval()
scheduler = DDIMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
print(f"Componentes cargados en {time.time() - t0:.0f} s")

def n_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"\n{'Componente':<22}{'Parámetros':>15}")
for name, m in [("CLIP Text Encoder", text_encoder), ("U-Net", unet),
                ("VAE (encoder)", vae.encoder), ("VAE (decoder)", vae.decoder)]:
    print(f"{name:<22}{n_params(m) / 1e6:>13.1f} M")
VAE_SCALE = vae.config.scaling_factor
print(f"\nFactor de escala del VAE: {VAE_SCALE}  (normaliza los latentes a varianza ≈ 1)")

## 2. Componente 1 — Del texto a los embeddings (Tokenizer + CLIP)

El *prompt* se convierte en IDs de tokens, siempre rellenados a **77 posiciones** (`<|startoftext|>`, los tokens del texto, `<|endoftext|>` y relleno). CLIP produce un vector de 768 dimensiones **por token**: ese tensor de 1×77×768 es lo único que la U-Net “sabe” del texto.

Para la *classifier-free guidance* (CFG) también codificamos un *prompt* vacío `""`: el embedding **incondicional**.

In [ ]:
prompt = "a watercolor painting of a red fox sitting in a snowy forest at sunrise, highly detailed"
negative_prompt = ""      # prompt "incondicional" para la guía libre de clasificador

def encode_text(text):
    tok = tokenizer(text, padding="max_length", max_length=tokenizer.model_max_length,
                    truncation=True, return_tensors="pt")
    emb = text_encoder(tok.input_ids.to(device))[0]          # last_hidden_state
    return tok.input_ids[0], emb

ids_cond, emb_cond = encode_text(prompt)
ids_uncond, emb_uncond = encode_text(negative_prompt)

tokens = tokenizer.convert_ids_to_tokens(ids_cond.tolist())
n_real = int((ids_cond != ids_cond[-1]).sum()) + 1          # tokens hasta el primer <|endoftext|>
print("IDs de tokens:", ids_cond.shape, "->", ids_cond[:n_real + 2].tolist(), "...")
print("Tokens:", tokens[:n_real + 1])
print("Embedding condicional:  ", tuple(emb_cond.shape), "  [batch, tokens, dimensión]")
print("Embedding incondicional:", tuple(emb_uncond.shape))

In [ ]:
E = emb_cond[0].float().cpu().numpy()
fig, ax = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [3, 1]})
im = ax[0].imshow(E[: n_real + 3].T[:128], aspect="auto", cmap="coolwarm", vmin=-2, vmax=2)
ax[0].set_xticks(range(n_real + 3))
ax[0].set_xticklabels([t.replace("</w>", "") for t in tokens[: n_real + 3]], rotation=70, fontsize=8)
ax[0].set_ylabel("primeras 128 de 768 dimensiones")
ax[0].set_title("Embeddings de CLIP por token (1×77×768)")
plt.colorbar(im, ax=ax[0])

# Similitud coseno entre el embedding condicional e incondicional, token a token
Eu = emb_uncond[0].float().cpu().numpy()
cos = (E * Eu).sum(1) / (np.linalg.norm(E, axis=1) * np.linalg.norm(Eu, axis=1))
ax[1].plot(cos, ".-"); ax[1].axvline(n_real - 1, color="r", ls="--", lw=1)
ax[1].set_xlabel("posición del token"); ax[1].set_ylabel("similitud coseno")
ax[1].set_title("cond vs incond"); ax[1].grid(True)
plt.tight_layout(); plt.show()

Fíjate en la gráfica de la derecha: la posición 0 (`<|startoftext|>`) es idéntica en ambos casos, y las posiciones de relleno también se parecen mucho. La información del *prompt* se concentra en las primeras posiciones, pero por la atención causal de CLIP **se filtra también a los tokens de relleno**, que la U-Net igual lee.

## 3. Componente 2 — El VAE: espacio de píxeles vs. espacio latente

Este es el corazón de la idea de *Latent Diffusion* (Rombach et al., 2022): **no hacer la difusión sobre los píxeles**, sino sobre una versión comprimida. Tomamos una imagen real de 512×512, la pasamos por el **encoder** del VAE y vemos qué queda.

- Píxeles: $3 \times 512 \times 512 = 786.432$ valores.
- Latente: $4 \times 64 \times 64 = 16.384$ valores → **48 veces menos**. Cada “píxel latente” resume un parche de 8×8 píxeles.

In [ ]:
from skimage import data

# Imagen de prueba incluida en scikit-image (un gato), recortada a cuadrado y llevada a 512x512
img = Image.fromarray(data.chelsea())
w, h = img.size; s = min(w, h)
img = img.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2)).resize((512, 512), Image.BICUBIC)

x = torch.from_numpy(np.array(img)).permute(2, 0, 1)[None].float() / 127.5 - 1   # [1,3,512,512] en [-1,1]
x = x.to(device)

posterior = vae.encode(x).latent_dist          # el encoder devuelve una distribución gaussiana
z = posterior.mean * VAE_SCALE                 # usamos la media, escalada (así entra a la U-Net)
print("Imagen en píxeles :", tuple(x.shape), f"= {x.numel():,} valores")
print("Latente           :", tuple(z.shape), f"= {z.numel():,} valores")
print(f"Compresión        : {x.numel() / z.numel():.0f}×")
print(f"Latente escalado  : media={z.mean():.3f}, desv. est.={z.std():.3f}")

x_rec = vae.decode(z / VAE_SCALE).sample.clamp(-1, 1)
mse = torch.mean(((x_rec - x) / 2) ** 2).item()
print(f"Reconstrucción    : PSNR = {10 * math.log10(1 / mse):.1f} dB")

In [ ]:
def to_img(t):
    # [1,3,H,W] en [-1,1] -> arreglo HxWx3 uint8
    return ((t[0].float().clamp(-1, 1).permute(1, 2, 0).cpu().numpy() + 1) * 127.5).round().astype(np.uint8)

# Aproximación lineal latente -> RGB (coeficientes conocidos para SD 1.x, usados en vistas previas rápidas)
LATENT_RGB = torch.tensor([[0.3512, 0.2297, 0.3227],
                           [0.3250, 0.4974, 0.2350],
                           [-0.2829, 0.1762, 0.2721],
                           [-0.2120, -0.2616, -0.7177]])

def latent_preview(z):
    # Proyecta los 4 canales latentes a 3 canales RGB (64x64) — sin usar el decoder
    rgb = torch.einsum("chw,cr->rhw", z[0].float().cpu(), LATENT_RGB)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    return rgb.permute(1, 2, 0).numpy()

zc = z[0].float().cpu().numpy()
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes[0, 0].imshow(np.array(img)); axes[0, 0].set_title("Píxeles: 3×512×512")
axes[0, 1].imshow(latent_preview(z), interpolation="nearest"); axes[0, 1].set_title("Latente 4×64×64 → RGB aprox.\n(sin decoder, 64×64 ampliado)")
axes[0, 2].imshow(to_img(x_rec)); axes[0, 2].set_title("Decodificado por el VAE: 3×512×512")
err = np.abs(to_img(x_rec).astype(float) - np.array(img).astype(float)).mean(2)
im = axes[0, 3].imshow(err, cmap="magma"); axes[0, 3].set_title("Error absoluto de reconstrucción")
plt.colorbar(im, ax=axes[0, 3], fraction=0.046)
for c in range(4):
    im = axes[1, c].imshow(zc[c], cmap="RdBu_r", interpolation="nearest")
    axes[1, c].set_title(f"Canal latente {c} (64×64)")
    plt.colorbar(im, ax=axes[1, c], fraction=0.046)
for a in axes.ravel():
    a.axis("off")
plt.suptitle("El mismo contenido en el espacio de píxeles y en el espacio latente", fontsize=14)
plt.tight_layout(); plt.show()

Observaciones: los 4 canales latentes siguen siendo **espacialmente coherentes** (se reconoce la silueta del gato en 64×64), pero no son colores: cada canal codifica una mezcla de luminancia, color y textura. El decoder recupera los detalles finos (pelos, bordes) que no caben en 64×64; por eso la reconstrucción es casi perfecta, con errores concentrados en las texturas de alta frecuencia.

In [ ]:
# Zoom: un parche de 8x8 en el latente corresponde a 64x64 en píxeles
r0, c0 = 24, 24
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(latent_preview(z)[r0:r0 + 8, c0:c0 + 8], interpolation="nearest")
ax[0].set_title("Latente: parche 8×8 (= 8×8×4 valores)")
ax[1].imshow(np.array(img)[r0 * 8:(r0 + 8) * 8, c0 * 8:(c0 + 8) * 8])
ax[1].set_title("Píxeles: la misma región, 64×64×3")
for a in ax: a.axis("off")
plt.show()

plt.figure(figsize=(7, 3))
for c in range(4):
    plt.hist(zc[c].ravel(), bins=80, alpha=0.5, label=f"canal {c}")
plt.title("Distribución de los valores latentes (escalados)"); plt.legend(); plt.grid(True); plt.show()

### Ruido en el espacio latente
La difusión directa se aplica **al latente**, no a la imagen. Aquí ensuciamos $z$ a distintos $t$ con la fórmula $z_t=\sqrt{\bar\alpha_t}\,z_0+\sqrt{1-\bar\alpha_t}\,\varepsilon$ y decodificamos cada uno: así “ve” la U-Net sus entradas durante el entrenamiento.

In [ ]:
ac = scheduler.alphas_cumprod
eps = torch.randn_like(z)
ts = [0, 100, 250, 400, 550, 700, 850, 999]
fig, axes = plt.subplots(2, len(ts), figsize=(2.2 * len(ts), 4.6))
for i, t in enumerate(ts):
    zt = ac[t].sqrt() * z + (1 - ac[t]).sqrt() * eps
    axes[0, i].imshow(latent_preview(zt), interpolation="nearest"); axes[0, i].set_title(f"t={t}", fontsize=9)
    axes[1, i].imshow(to_img(vae.decode(zt / VAE_SCALE).sample))
    axes[0, i].axis("off"); axes[1, i].axis("off")
axes[0, 0].set_ylabel("latente"); axes[1, 0].set_ylabel("decodificado")
plt.suptitle("Arriba: z_t (64×64, vista RGB aprox.) · Abajo: VAE.decode(z_t) en 512×512")
plt.tight_layout(); plt.show()

## 4. Componente 3 — Dentro de la U-Net

La U-Net recibe tres cosas: el latente ruidoso $z_t$ (B×4×64×64), el paso $t$ y los embeddings del texto (B×77×768). Registramos *hooks* de PyTorch en cada bloque para ver **cómo cambia la forma de los tensores** a medida que bajan y suben por la red. Los bloques `CrossAttn…` son los que leen el texto.

In [ ]:
shapes = []
def make_hook(name):
    def hook(module, inp, out):
        o = out[0] if isinstance(out, tuple) else out
        shapes.append((name, type(module).__name__, tuple(o.shape)))
    return hook

handles = []
for i, b in enumerate(unet.down_blocks):
    handles.append(b.register_forward_hook(make_hook(f"down_blocks[{i}]")))
handles.append(unet.mid_block.register_forward_hook(make_hook("mid_block")))
for i, b in enumerate(unet.up_blocks):
    handles.append(b.register_forward_hook(make_hook(f"up_blocks[{i}]")))

z_in = torch.randn(1, 4, 64, 64, device=device, dtype=DTYPE)
out = unet(z_in, 500, encoder_hidden_states=emb_cond.to(DTYPE)).sample
for h in handles:
    h.remove()

print(f"{'entrada z_t':<16}{'':<28}{str((1, 4, 64, 64)):>22}")
for name, cls, shp in shapes:
    print(f"{name:<16}{cls:<28}{str(shp):>22}")
print(f"{'salida ε̂':<16}{'':<28}{str(tuple(out.shape)):>22}")

La resolución espacial baja 64→32→16→8 mientras los canales suben 320→640→1280, y luego se recorre el camino inverso. **La salida tiene exactamente la misma forma que la entrada (1×4×64×64)**, porque la U-Net predice el ruido $\hat\varepsilon$ que hay que quitarle al latente.

## 5. Componente 4 — El scheduler

El scheduler define, para cada uno de los 1000 pasos de entrenamiento, cuánta señal queda ($\bar\alpha_t$) y cómo dar un paso de *denoising*. Usamos **DDIM** con 50 pasos: en vez de recorrer los 1000 pasos visita 50 de ellos ($t$ = 981, 961, …, 1).

En cada paso DDIM hace dos cosas: (1) estima la imagen limpia $\hat z_0=(z_t-\sqrt{1-\bar\alpha_t}\,\hat\varepsilon)/\sqrt{\bar\alpha_t}$ y (2) la vuelve a ensuciar al nivel de $t-\Delta$.

> **Nota sobre el notebook de clase:** allí la actualización es `latents = latents - step_size * noise_pred`, que no corresponde a ningún scheduler de difusión (no usa $\bar\alpha_t$), por eso la imagen final puede salir “lavada”. Aquí usamos `scheduler.step(...)`, que aplica la ecuación correcta.

In [ ]:
NUM_STEPS = 50
scheduler.set_timesteps(NUM_STEPS)
print("Primeros pasos:", scheduler.timesteps[:6].tolist(), "...", "Últimos:", scheduler.timesteps[-3:].tolist())

plt.figure(figsize=(7, 3))
plt.plot(ac.numpy(), label=r"$\bar\alpha_t$ (señal)")
plt.plot((1 - ac).numpy(), label=r"$1-\bar\alpha_t$ (ruido)")
plt.scatter(scheduler.timesteps.numpy(), ac[scheduler.timesteps].numpy(), s=10, c="k", label="pasos DDIM visitados")
plt.xlabel("t"); plt.legend(); plt.grid(True); plt.title("Scheduler 'scaled_linear' de SD 1.5")
plt.show()

## 6. El ciclo de generación, escrito a mano

En cada paso:
1. Duplicamos el latente (una copia para el texto, otra para el *prompt* vacío).
2. La U-Net predice el ruido para ambas: $\hat\varepsilon_{uncond}$ y $\hat\varepsilon_{text}$.
3. **Classifier-free guidance:** $\hat\varepsilon=\hat\varepsilon_{uncond}+w\,(\hat\varepsilon_{text}-\hat\varepsilon_{uncond})$, con $w$ = 7.5.
4. El scheduler calcula $z_{t-1}$ y la estimación $\hat z_0$.

Guardamos todo en cada paso para visualizarlo después.

In [ ]:
GUIDANCE = 7.5
SEED = 1234
HEIGHT = WIDTH = 512

def generate(emb_c, emb_u, guidance=GUIDANCE, steps=NUM_STEPS, seed=SEED, record=True):
    scheduler.set_timesteps(steps)
    g = torch.Generator(device="cpu").manual_seed(seed)
    latents = torch.randn((1, unet.config.in_channels, HEIGHT // 8, WIDTH // 8), generator=g).to(device)
    latents = latents * scheduler.init_noise_sigma
    context = torch.cat([emb_u, emb_c]).to(DTYPE)            # [2, 77, 768]
    hist = {"t": [], "z_t": [], "z0_hat": [], "eps_u_norm": [], "eps_c_norm": [], "guid_norm": [], "z_std": []}
    for t in scheduler.timesteps:
        model_in = scheduler.scale_model_input(torch.cat([latents] * 2), t)
        eps_u, eps_c = unet(model_in.to(DTYPE), t, encoder_hidden_states=context).sample.float().chunk(2)
        eps = eps_u + guidance * (eps_c - eps_u)              # classifier-free guidance
        step = scheduler.step(eps, t, latents)
        if record:
            hist["t"].append(int(t))
            hist["z_t"].append(latents.cpu())
            hist["z0_hat"].append(step.pred_original_sample.cpu())
            hist["eps_u_norm"].append(eps_u.norm().item())
            hist["eps_c_norm"].append(eps_c.norm().item())
            hist["guid_norm"].append((eps_c - eps_u).norm().item())
            hist["z_std"].append(latents.std().item())
        latents = step.prev_sample
    return latents, hist

@torch.no_grad()
def decode(latents):
    return to_img(vae.decode(latents.to(device).float() / VAE_SCALE).sample)

t0 = time.time()
z_final, hist = generate(emb_cond, emb_uncond)
image_final = decode(z_final)
print(f"Generación: {time.time() - t0:.1f} s | latente final {tuple(z_final.shape)} → imagen {image_final.shape}")

### 6.1 Resultado: el latente 64×64 frente a la imagen 512×512

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 6))
ax[0].imshow(latent_preview(z_final), interpolation="nearest")
ax[0].set_title("Latente final z₀: 4×64×64\n(vista RGB aproximada, píxeles ampliados 8×)")
zf = z_final[0].float().cpu().numpy()
mosaic = np.block([[zf[0], zf[1]], [zf[2], zf[3]]])
ax[1].imshow(mosaic, cmap="RdBu_r", interpolation="nearest")
ax[1].set_title("Sus 4 canales (cada uno 64×64)")
ax[2].imshow(image_final)
ax[2].set_title("VAE.decode(z₀): 3×512×512")
for a in ax: a.axis("off")
plt.suptitle(f'"{prompt}"', fontsize=11); plt.tight_layout(); plt.show()

### 6.2 El proceso de generación en el espacio latente
Cada fila es un canal latente; cada columna, un paso. Los primeros pasos son ruido puro; hacia la mitad aparece la estructura, y al final los canales quedan casi fijos.

In [ ]:
idx = np.linspace(0, NUM_STEPS - 1, 10).round().astype(int)
fig, axes = plt.subplots(5, len(idx), figsize=(1.8 * len(idx), 9.5))
for j, k in enumerate(idx):
    zt = hist["z_t"][k]
    axes[0, j].imshow(latent_preview(zt), interpolation="nearest")
    axes[0, j].set_title(f"paso {k + 1}\nt={hist['t'][k]}", fontsize=8)
    for c in range(4):
        axes[c + 1, j].imshow(zt[0, c].numpy(), cmap="RdBu_r", vmin=-3, vmax=3, interpolation="nearest")
for a in axes.ravel(): a.axis("off")
for c, lab in enumerate(["RGB aprox."] + [f"canal {i}" for i in range(4)]):
    axes[c, 0].text(-12, 32, lab, rotation=90, va="center", ha="right", fontsize=9)
plt.suptitle("Evolución de z_t (1×4×64×64) a lo largo de la generación"); plt.tight_layout(); plt.show()

### 6.3 El mismo proceso visto en píxeles
Decodificamos con el VAE dos tensores en cada paso: el latente ruidoso **$z_t$** (lo que la U-Net recibe) y la **predicción $\hat z_0$** (lo que la U-Net “cree” que será la imagen final). La segunda fila muestra que la composición se decide muy temprano, aunque $z_t$ todavía sea casi todo ruido.

In [ ]:
fig, axes = plt.subplots(2, len(idx), figsize=(2.0 * len(idx), 4.6))
for j, k in enumerate(idx):
    axes[0, j].imshow(decode(hist["z_t"][k])); axes[0, j].set_title(f"t={hist['t'][k]}", fontsize=8)
    axes[1, j].imshow(decode(hist["z0_hat"][k]))
for a in axes.ravel(): a.axis("off")
axes[0, 0].text(-40, 256, "decode(z_t)", rotation=90, va="center", ha="right")
axes[1, 0].text(-40, 256, "decode(ẑ₀)", rotation=90, va="center", ha="right")
plt.suptitle("Proceso de generación decodificado a 512×512"); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].plot(hist["t"], hist["eps_u_norm"], label="‖ε̂ incondicional‖")
ax[0].plot(hist["t"], hist["eps_c_norm"], "--", label="‖ε̂ con texto‖")
ax[0].plot(hist["t"], np.array(hist["guid_norm"]) * GUIDANCE, label=f"‖w·(ε̂_text − ε̂_uncond)‖, w={GUIDANCE}")
ax[0].invert_xaxis(); ax[0].set_xlabel("t (de 1000 a 0 →)"); ax[0].legend(fontsize=8); ax[0].grid(True)
ax[0].set_title("Magnitud del ruido predicho y de la guía del texto")
ax[1].plot(hist["t"], hist["z_std"]); ax[1].invert_xaxis(); ax[1].grid(True)
ax[1].set_xlabel("t"); ax[1].set_title("Desviación estándar de z_t")
plt.tight_layout(); plt.show()

### 6.4 Animación del proceso
Guardamos un GIF que muestra, cuadro por cuadro, el latente (izquierda, 64×64 ampliado) y su decodificación $\hat z_0$ (derecha, 512×512).

In [ ]:
from IPython.display import Image as IPImage, display

frames = []
for k in range(0, NUM_STEPS, 2):
    lat = (latent_preview(hist["z_t"][k]) * 255).astype(np.uint8)
    lat = np.array(Image.fromarray(lat).resize((512, 512), Image.NEAREST))
    frames.append(Image.fromarray(np.concatenate([lat, decode(hist["z0_hat"][k])], axis=1)))
frames += [Image.fromarray(np.concatenate(
    [np.array(Image.fromarray((latent_preview(z_final) * 255).astype(np.uint8)).resize((512, 512), Image.NEAREST)),
     image_final], axis=1))] * 6
frames = [f.resize((512, 256)) for f in frames]
frames[0].save("proceso_stable_diffusion.gif", save_all=True, append_images=frames[1:], duration=150, loop=0)
display(IPImage("proceso_stable_diffusion.gif"))

## 7. Interacción texto ↔ imagen: mapas de atención cruzada

¿Cómo “sabe” la U-Net dónde poner al zorro? En sus capas de **atención cruzada** (`attn2`), cada posición espacial del latente (consulta, *query*) se compara con los 77 tokens del texto (claves, *keys*). Reemplazamos el procesador de esas capas por uno propio que **guarda las probabilidades de atención**, las promediamos sobre cabezas, capas y pasos a resolución 16×16, y las proyectamos sobre la imagen final.

In [ ]:
class StoreCrossAttn(AttnProcessor):
    # Igual al procesador estándar de diffusers, pero guarda las probabilidades de atención cruzada
    store = []
    def __call__(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, temb=None, *args, **kwargs):
        batch, seq_len, _ = hidden_states.shape
        query = attn.head_to_batch_dim(attn.to_q(hidden_states))
        key = attn.head_to_batch_dim(attn.to_k(encoder_hidden_states))
        value = attn.head_to_batch_dim(attn.to_v(encoder_hidden_states))
        probs = attn.get_attention_scores(query, key, None)          # [batch*heads, HW, 77]
        side = int(math.sqrt(seq_len))
        if side == 16:                                               # nos quedamos con la resolución 16x16
            p = probs.reshape(batch, attn.heads, seq_len, -1)[batch // 2:]   # sólo la mitad condicionada por texto
            StoreCrossAttn.store.append(p.mean(1)[0].float().cpu())  # promedio de cabezas -> [256, 77]
        out = attn.batch_to_head_dim(torch.bmm(probs, value))
        return attn.to_out[1](attn.to_out[0](out))

original_procs = unet.attn_processors
unet.set_attn_processor({name: (StoreCrossAttn() if "attn2" in name else proc)
                         for name, proc in original_procs.items()})
StoreCrossAttn.store = []
z_att, _ = generate(emb_cond, emb_uncond, record=False)
unet.set_attn_processor(original_procs)                              # restauramos los procesadores originales

A = torch.stack(StoreCrossAttn.store).mean(0)                        # [256, 77]
print("Mapas guardados:", len(StoreCrossAttn.store), "| forma promedio:", tuple(A.shape))

In [ ]:
img_att = decode(z_att)
words = [(i, t.replace("</w>", "")) for i, t in enumerate(tokens[1:n_real - 1], start=1)]   # sin <|startoftext|> ni <|endoftext|>
n_cols = 6
n_rows = math.ceil((len(words) + 1) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.6 * n_cols, 2.8 * n_rows))
axes = axes.ravel()
axes[0].imshow(img_att); axes[0].set_title("imagen", fontsize=9)
for a_i, (tok_i, word) in enumerate(words, start=1):
    m = A[:, tok_i].reshape(16, 16).numpy()
    m = np.array(Image.fromarray((m / m.max() * 255).astype(np.uint8)).resize((512, 512), Image.BICUBIC)) / 255
    axes[a_i].imshow(img_att); axes[a_i].imshow(m, cmap="jet", alpha=0.5)
    axes[a_i].set_title(word, fontsize=10)
for a in axes: a.axis("off")
plt.suptitle("Atención cruzada promedio por token (16×16 → 512×512)"); plt.tight_layout(); plt.show()

Las palabras que nombran objetos (por ejemplo *fox*, *forest*, *snowy*) “encienden” regiones concretas de la imagen: así el texto llega al lugar correcto del latente. Las palabras de estilo (*watercolor*, *detailed*) suelen repartir su atención por toda la imagen.

## 8. Efecto de la *classifier-free guidance*
Mismo ruido inicial y mismo *prompt*, variando solo la escala $w$. Con $w=1$ el texto apenas guía (la imagen es poco fiel al *prompt*); con valores muy altos la imagen se satura.

In [ ]:
scales = [1.0, 3.0, 7.5, 12.0, 20.0]
fig, axes = plt.subplots(1, len(scales), figsize=(3.2 * len(scales), 3.6))
for a, w in zip(axes, scales):
    zw, _ = generate(emb_cond, emb_uncond, guidance=w, record=False)
    a.imshow(decode(zw)); a.set_title(f"w = {w}"); a.axis("off")
plt.suptitle("Escala de guía (CFG)"); plt.tight_layout(); plt.show()

## 9. Resumen de los tensores que viajan por Stable Diffusion

| Paso | Tensor | Forma | Espacio |
|---|---|---|---|
| Tokenizer | `input_ids` | 1×77 | enteros (vocabulario de 49.408) |
| Text Encoder | `encoder_hidden_states` | 1×77×768 (2×77×768 con CFG) | semántico (CLIP) |
| Ruido inicial | $z_T$ | 1×4×64×64 | **latente** |
| U-Net (por paso) | entrada / salida $\hat\varepsilon$ | 2×4×64×64 → 2×4×64×64 | **latente** |
| U-Net (interno) | mapas de características | hasta 1×1280×8×8 | latente profundo |
| Scheduler | $z_{t-1}$, $\hat z_0$ | 1×4×64×64 | **latente** |
| VAE Decoder | imagen | 1×3×512×512 | **píxeles** |

**Conclusión.** Las 50 iteraciones de la U-Net (la parte cara) ocurren sobre un tensor 48 veces más pequeño que la imagen, y el VAE solo se usa una vez al final. Esa es la razón por la que Stable Diffusion puede correr en una GPU de consumo. El texto nunca toca los píxeles: entra únicamente a través de la atención cruzada de la U-Net, y la guía libre de clasificador amplifica su efecto.